# Tóm tắt văn bản

Notebook xử lý độc lập từng topic trong `data/DUC_TEXT/train`. 

Kết quả giữ nguyên thẻ `<s ...>...</s>` và được ghi vào `data/output/duc-textrank-pipeline`.

## 1. Thiết lập

Notebook dùng các thư viện quen thuộc và gom tham số tại một nơi. Mỗi bước được viết bằng biến trung gian và vòng lặp để người mới dễ theo dõi.

In [1]:
import math
import re
from pathlib import Path
from bs4 import BeautifulSoup

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

NOTEBOOK_SLUG = 'duc-textrank-pipeline'
INPUT_DIR = PROJECT_ROOT / 'data' / 'DUC_TEXT' / 'train'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'output' / NOTEBOOK_SLUG

SIMILARITY_THRESHOLD = 0.0125
PAGERANK_DAMPING = 0.85
PAGERANK_TOLERANCE = 1e-8
PAGERANK_MAX_ITERATIONS = 1000
MAX_SUMMARY_WORDS = 100
USE_MMR = True
MMR_LAMBDA = 0.70

assert INPUT_DIR.is_dir(), f'Không tìm thấy thư mục input: {INPUT_DIR}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Dir input : {INPUT_DIR}')
print(f'Dir output: {OUTPUT_DIR}')

Dir input : /Users/thangtran/Workplace/master_s_degree/nlp/nlp-practice/data/DUC_TEXT/train
Dir output: /Users/thangtran/Workplace/master_s_degree/nlp/nlp-practice/data/output/duc-textrank-pipeline


## 2. Parse topic và tiền xử lý câu

BeautifulSoup tìm từng thẻ `<s>`. Mỗi câu được lưu bằng một `dict` gồm nội dung, token và vị trí trong topic. Cách này tránh class, kế thừa và `dataclass`.

In [ ]:
# Sử dụng thư viện

STOP_WORDS = {
    'a', 'an', 'and', 'are', 'as', 'at', 'be', 'been', 'being', 'but', 'by',
    'for', 'from', 'had', 'has', 'have', 'he', 'her', 'hers', 'him', 'his',
    'i', 'if', 'in', 'into', 'is', 'it', 'its', 'no', 'not', 'of', 'on',
    'or', 'our', 'ours', 'she', 'so', 'that', 'the', 'their', 'theirs',
    'them', 'there', 'they', 'this', 'those', 'to', 'was', 'we', 'were',
    'what', 'when', 'where', 'which', 'who', 'will', 'with', 'would', 'you',
    'your', 'said', 'about', 'after', 'all', 'also', 'before', 'could',
    'more', 'most', 'other', 'over', 'than', 'then', 'up'
}

TOKEN_PATTERN = re.compile(r"[a-z]+(?:'[a-z]+)?|\d+")

def preprocess(text):
    all_tokens = TOKEN_PATTERN.findall(text.lower())

    filtered_tokens = []
    for token in all_tokens:
        if token not in STOP_WORDS:
            filtered_tokens.append(token)

    return filtered_tokens


def read_topic(topic_path):
    file_content = topic_path.read_text(encoding='utf-8')
    soup = BeautifulSoup(file_content, 'html.parser')
    sentences = []

    for sentence_tag in soup.find_all('s'):
        content = sentence_tag.get_text().strip()
        tokens = preprocess(content)

        if len(tokens) == 0:
            continue

        sentence = {}
        sentence['docid'] = str(sentence_tag.get('docid', ''))
        sentence['num'] = int(sentence_tag.get('num', 0)) # type: ignore
        sentence['wdcount'] = int(sentence_tag.get('wdcount', len(content.split()))) # type: ignore
        sentence['content'] = content
        sentence['tagged_content'] = str(sentence_tag)
        sentence['tokens'] = tokens
        sentence['source_index'] = len(sentences)
        sentences.append(sentence)

    return sentences

## 3. Tự tính TF-IDF

**TF được tính bằng**:

$$
TF(t,d)=\frac{count(t,d)}{|d|}
$$

**IDF được tính bằng Smooth IDF**:

$$
IDF(t)=\log\left(\frac{1+N}{1+df(t)}\right)+1
$$

*Trong đó*:

- $N$ là tổng số câu trong corpus.
- $df(t)$ là số câu chứa term $t$.

**Trọng số TF-IDF được tính bằng**:

$$
TFIDF(t,d)=TF(t,d)\times IDF(t)
$$

**Sau đó vector TF-IDF được chuẩn hóa L2**.

**L2 norm**:

$$
\lVert v\rVert_2
=
\sqrt{
w_1^2+w_2^2+\cdots+w_n^2
}
$$

**Mỗi trọng số được chuẩn hóa**:

$$
\hat{w}_i
=
\frac{w_i}{\lVert v\rVert_2}
$$

**Sau chuẩn hóa**:

$$
\lVert \hat{v}\rVert_2 = 1
$$

In [16]:
def calculate_tf(tokens):
    counts = {}
    for token in tokens:
        if token not in counts:
            counts[token] = 0
        counts[token] = counts[token] + 1

    token_count = len(tokens)
    tf = {}
    for term, count in counts.items():
        tf[term] = count / token_count
    return tf


def calculate_idf(tokenized_sentences):
    sentence_count = len(tokenized_sentences)
    document_frequency = {}

    for tokens in tokenized_sentences:
        unique_tokens = set(tokens)
        for term in unique_tokens:
            if term not in document_frequency:
                document_frequency[term] = 0
            document_frequency[term] = document_frequency[term] + 1

    idf = {}
    for term, frequency in document_frequency.items():
        numerator = 1 + sentence_count
        denominator = 1 + frequency
        idf[term] = math.log(numerator / denominator) + 1
    return idf


def l2_normalize(vector):
    squared_sum = 0.0
    for weight in vector.values():
        squared_sum = squared_sum + weight * weight
    norm = math.sqrt(squared_sum)
    if norm == 0.0:
        return {}

    normalized_vector = {}
    for term, weight in vector.items():
        normalized_vector[term] = weight / norm
    return normalized_vector


def calculate_tfidf_vectors(sentences):
    tokenized_sentences = []
    for sentence in sentences:
        tokenized_sentences.append(sentence['tokens'])

    idf = calculate_idf(tokenized_sentences)
    vectors = []
    for tokens in tokenized_sentences:
        tf = calculate_tf(tokens)
        vector = {}
        for term, tf_value in tf.items():
            vector[term] = tf_value * idf[term]
        vectors.append(l2_normalize(vector))
    return vectors

## 4. Cosine similarity và đồ thị câu

Vì các vector đã được L2-normalized ở bước trước, nên:

$$\lVert q \rVert_2 = 1,\qquad \lVert d \rVert_2 = 1$$

Công thức cosine đầy đủ là:

$$\cos(q,d)=\frac{q\cdot d}{\lVert q\rVert_2\lVert d\rVert_2}$$

Do hai norm đều bằng 1, ta có:

$$\cos(q,d)=q\cdot d$$

In [4]:
def cosine_similarity(left_vector, right_vector):
    similarity = 0.0
    for term, left_weight in left_vector.items():
        right_weight = right_vector.get(term, 0.0)
        similarity = similarity + left_weight * right_weight
    return similarity


def build_sentence_graph(vectors, similarity_threshold):
    sentence_count = len(vectors)

    graph = {}
    for index in range(sentence_count):
        graph[index] = {}

    similarities = []
    for row_index in range(sentence_count):
        row = [0.0] * sentence_count
        similarities.append(row)

    for left_index in range(sentence_count):
        for right_index in range(left_index + 1, sentence_count):
            similarity = cosine_similarity(vectors[left_index], vectors[right_index])
            similarities[left_index][right_index] = similarity
            similarities[right_index][left_index] = similarity
            if similarity >= similarity_threshold:
                graph[left_index][right_index] = similarity
                graph[right_index][left_index] = similarity

    return graph, similarities

## 5. Tự cài đặt PageRank

Ban đầu, mỗi node được gán PageRank bằng nhau:

$$
PR_0(i)=\frac{1}{N}
$$

Trong đó $N$ là tổng số node trong đồ thị.

Tổng trọng số các cạnh đi ra từ node $j$:

$$
W_j=\sum_{k \in Out(j)} w_{jk}
$$

Trong đó $w_{jk}$ là trọng số cạnh từ node $j$ đến node $k$.

Với các node không có cạnh (dangling node), tổng PageRank của chúng được tính bằng:

$$
D=\sum_{j:W_j=0} PR(j)
$$

Ở mỗi vòng lặp, PageRank mới của node $i$ được tính bằng:

$$
PR_{new}(i)
=
\frac{1-d}{N}
+
d\frac{D}{N}
+
d
\sum_{j:i\in Out(j)}
PR(j)\frac{w_{ji}}{W_j}
$$

Trong đó:

- $d$ là damping factor, mặc định $d=0.85$.
- $\frac{1-d}{N}$ là phần điểm cơ sở được chia đều cho tất cả node.
- $d\frac{D}{N}$ là PageRank từ các dangling node được chia đều cho tất cả node.
- $PR(j)\frac{w_{ji}}{W_j}$ là phần PageRank mà node $j$ truyền cho node $i$ theo tỷ lệ trọng số cạnh.

Sau mỗi vòng lặp, độ thay đổi giữa hai vector PageRank được tính bằng:

$$
\Delta
=
\sum_{i=1}^{N}
\left|
PR_{new}(i)-PR(i)
\right|
$$

Thuật toán được xem là hội tụ khi:

$$
\Delta < \varepsilon
$$

Trong đó $\varepsilon$ là tolerance, mặc định:

$$
\varepsilon=10^{-8}
$$

Sau khi hội tụ, tổng PageRank của tất cả các node phải xấp xỉ bằng 1:

$$
\sum_{i=1}^{N}PR(i)\approx1
$$

In [5]:
def calculate_pagerank(
    graph,
    damping=0.85,
    tolerance=1e-8,
    max_iterations=1000,
):
    node_count = len(graph)
    if node_count == 0:
        return {}, 0
    if not 0.0 < damping < 1.0:
        raise ValueError('damping phải nằm trong khoảng (0, 1)')

    scores = {}
    outgoing_weights = {}
    for node, neighbors in graph.items():
        scores[node] = 1.0 / node_count
        outgoing_weights[node] = sum(neighbors.values())

    for iteration in range(1, max_iterations + 1):
        dangling_score = 0.0
        for node, total_weight in outgoing_weights.items():
            if total_weight == 0.0:
                dangling_score = dangling_score + scores[node]

        base_score = (1.0 - damping) / node_count
        dangling_share = damping * dangling_score / node_count

        new_scores = {}
        for node in graph:
            new_scores[node] = base_score + dangling_share

        for source, neighbors in graph.items():
            total_weight = outgoing_weights[source]
            if total_weight == 0.0:
                continue
            for target, edge_weight in neighbors.items():
                new_scores[target] += (
                    damping * scores[source] * edge_weight / total_weight
                )

        difference = 0.0
        for node in graph:
            score_change = abs(new_scores[node] - scores[node])
            difference = difference + score_change
        scores = new_scores
        if difference < tolerance:
            assert math.isclose(sum(scores.values()), 1.0, rel_tol=1e-9, abs_tol=1e-9)
            return scores, iteration

    raise RuntimeError(f'PageRank không hội tụ sau {max_iterations} vòng lặp')

## 6. Chọn câu bằng PageRank và MMR

Sau khi PageRank hội tụ, điểm PageRank được chuẩn hóa về khoảng $[0,1]$
bằng Min-Max Normalization:

$$
R(i)
=
\frac{PR(i)-PR_{min}}
{PR_{max}-PR_{min}}
$$

Trong đó:

- $PR(i)$ là điểm PageRank của câu $i$.
- $PR_{min}$ là PageRank nhỏ nhất.
- $PR_{max}$ là PageRank lớn nhất.
- $R(i)$ là relevance score sau chuẩn hóa.

Nếu tất cả câu có cùng PageRank:

$$
PR_{min}=PR_{max}
$$

thì:

$$
R(i)=1
$$

cho tất cả các câu.

---

Khi MMR được sử dụng, độ dư thừa của câu ứng viên $i$
so với tập câu đã chọn $S$ được tính bằng:

$$
Redundancy(i)
=
\max_{j \in S}
Sim(i,j)
$$

Trong đó $Sim(i,j)$ là Cosine Similarity giữa câu $i$ và câu $j$.

Nếu chưa có câu nào được chọn:

$$
Redundancy(i)=0
$$

Điểm MMR của câu ứng viên được tính bằng:

$$
MMR(i)
=
\lambda R(i)
-
(1-\lambda)Redundancy(i)
$$

hay:

$$
MMR(i)
=
\lambda R(i)
-
(1-\lambda)
\max_{j\in S}Sim(i,j)
$$

Trong đó:

- $\lambda$ điều khiển sự cân bằng giữa độ quan trọng và độ dư thừa.
- $R(i)$ là relevance score từ PageRank đã chuẩn hóa.
- $Sim(i,j)$ là Cosine Similarity giữa hai câu.

Khi $\lambda$ tiến gần $1$, thuật toán ưu tiên PageRank:

$$
\lambda \rightarrow 1
\Rightarrow
MMR(i) \approx R(i)
$$

Khi $\lambda$ giảm, thuật toán phạt mạnh hơn các câu giống với
những câu đã được chọn.

Ở mỗi vòng lặp, câu có MMR lớn nhất được chọn:

$$
i^*
=
\arg\max_{i \in C} MMR(i)
$$

với điều kiện tổng số từ không vượt quá giới hạn:

$$
W_{used}+W_i \le W_{max}
$$

Trong đó:

- $W_{used}$ là tổng số từ của các câu đã chọn.
- $W_i$ là số từ của câu ứng viên $i$.
- $W_{max}$ là giới hạn số từ của bản tóm tắt.

Quá trình tiếp tục cho đến khi không còn câu nào có thể được thêm
mà vẫn thỏa mãn giới hạn số từ.

Sau khi hoàn thành quá trình lựa chọn, các câu được sắp xếp lại
theo thứ tự xuất hiện ban đầu trước khi tạo văn bản tóm tắt.

In [6]:
def normalize_scores(scores):
    low = min(scores.values())
    high = max(scores.values())
    normalized_scores = {}

    if math.isclose(low, high):
        for index in scores:
            normalized_scores[index] = 1.0
        return normalized_scores

    for index, score in scores.items():
        normalized_scores[index] = (score - low) / (high - low)
    return normalized_scores


def select_sentences(
    sentences,
    pagerank_scores,
    similarities,
    max_words,
    use_mmr,
    mmr_lambda,
):
    relevance = normalize_scores(pagerank_scores)
    remaining = set(range(len(sentences)))
    selected = []
    used_words = 0

    while len(remaining) > 0:
        best_index = None
        best_comparison = None

        for index in remaining:
            sentence_words = len(sentences[index]['content'].split())
            if used_words + sentence_words > max_words:
                continue

            redundancy = 0.0
            if use_mmr and len(selected) > 0:
                for selected_index in selected:
                    similarity = similarities[index][selected_index]
                    if similarity > redundancy:
                        redundancy = similarity

            selection_score = relevance[index]
            if use_mmr:
                selection_score = (
                    mmr_lambda * relevance[index]
                    - (1.0 - mmr_lambda) * redundancy
                )

            comparison = (
                selection_score,
                pagerank_scores[index],
                -sentences[index]['source_index'],
            )
            if best_comparison is None or comparison > best_comparison:
                best_comparison = comparison
                best_index = index

        if best_index is None:
            break

        selected.append(best_index)
        remaining.remove(best_index)
        selected_word_count = len(sentences[best_index]['content'].split())
        used_words = used_words + selected_word_count

    return selected


def summarize_topic(topic_path):
    sentences = read_topic(topic_path)
    if not sentences:
        return '', {'sentences': 0, 'edges': 0, 'selected': 0, 'words': 0, 'iterations': 0}

    vectors = calculate_tfidf_vectors(sentences)
    graph, similarities = build_sentence_graph(vectors, SIMILARITY_THRESHOLD)
    scores, iterations = calculate_pagerank(
        graph, PAGERANK_DAMPING, PAGERANK_TOLERANCE, PAGERANK_MAX_ITERATIONS
    )
    selected = select_sentences(
        sentences, scores, similarities, MAX_SUMMARY_WORDS, USE_MMR, MMR_LAMBDA
    )
    selected.sort()

    summary_lines = []
    selected_word_total = 0
    for index in selected:
        summary_lines.append(sentences[index]['tagged_content'])
        sentence_words = len(sentences[index]['content'].split())
        selected_word_total = selected_word_total + sentence_words
    summary = '\n'.join(summary_lines)

    all_neighbor_count = 0
    for neighbors in graph.values():
        all_neighbor_count = all_neighbor_count + len(neighbors)
    edge_count = all_neighbor_count // 2
    return summary, {
        'sentences': len(sentences), 'edges': edge_count, 'selected': len(selected),
        'words': selected_word_total,
        'iterations': iterations,
    }

## 7. Chạy tập train và kiểm tra output

### 7.1 Chạy thử với 1 bài viết

In [ ]:
file_txt_path = Path.cwd().parent / "data" / "DUC_TEXT" / "train" / "d061j"

try:
    read_txt = file_txt_path.read_text(encoding="utf-8")
except FileNotFoundError:
    print("File not found")
else:
    print(f"Your file length: {len(read_txt)} characters")

summary, stats = summarize_topic(file_txt_path)

print(f"summary:\n{summary}")
print(f"stats: {stats}")


Your file length: 32406 characters
summary:
<s docid="AP880911-0016" num="14" wdcount="15"> Tropical Storm Gilbert formed in the eastern Caribbean and strengthened into a hurricane Saturday night.</s>
<s docid="AP880912-0095" num="34" wdcount="25"> Maximum sustained winds were near 110 mph, with tropical-storm force winds extending up to 250 miles to the north and 100 miles to the south.</s>
<s docid="AP880912-0137" num="18" wdcount="21"> A National Weather Service report said the hurricane was moving west at 17 mph with maximum sustained winds of 115 mph.</s>
<s docid="AP880912-0137" num="22" wdcount="13"> Gilbert reached Jamaica after skirting southern Puerto Rico, Haiti and the Dominican Republic.</s>
<s docid="WSJ880912-0064" num="11" wdcount="25"> At 3 p.m. EDT, the center of the hurricane was about 100 miles south of the Dominican Republic and 425 miles east of Kingston, Jamaica.</s>
stats: {'sentences': 186, 'edges': 820, 'selected': 5, 'words': 99, 'iterations': 50}


### 7.2 Chạy toàn bộ bài viết

In [ ]:
topic_paths = []
for path in INPUT_DIR.iterdir():
    if path.is_file():
        topic_paths.append(path)
topic_paths.sort()

run_stats = {}
for topic_path in topic_paths:
    summary, stats = summarize_topic(topic_path)
    (OUTPUT_DIR / topic_path.name).write_text(summary.strip() + '\n', encoding='utf-8')
    run_stats[topic_path.name] = stats

output_paths = []
for path in OUTPUT_DIR.iterdir():
    if path.is_file():
        output_paths.append(path)
output_paths.sort()

input_names = []
for path in topic_paths:
    input_names.append(path.name)

output_names = []
for path in output_paths:
    output_names.append(path.name)

assert output_names == input_names

for path in output_paths:
    output_text = path.read_text(encoding='utf-8').strip()
    assert output_text != ''
    for line in output_text.splitlines():
        assert line.startswith('<s ')
        assert line.endswith('</s>')

for stats in run_stats.values():
    assert stats['words'] <= MAX_SUMMARY_WORDS

word_counts = []
iteration_counts = []
for stats in run_stats.values():
    word_counts.append(stats['words'])
    iteration_counts.append(stats['iterations'])
    
print(f'Đã xử lý: {len(run_stats)} topic')
print(f'Số từ mỗi summary: min={min(word_counts)}, max={max(word_counts)}')
print(f'Số vòng PageRank: min={min(iteration_counts)}, max={max(iteration_counts)}')
print(f'Thư mục kết quả: {OUTPUT_DIR}')
print('\nVí dụ output d061j:')
print((OUTPUT_DIR / 'd061j').read_text(encoding='utf-8')[:700])

Đã xử lý: 50 topic
Số từ mỗi summary: min=98, max=100
Số vòng PageRank: min=35, max=87
Thư mục kết quả: /Users/thangtran/Workplace/master_s_degree/nlp/nlp-practice/data/output/duc-textrank-pipeline

Ví dụ output d061j:
<s docid="AP880911-0016" num="14" wdcount="15"> Tropical Storm Gilbert formed in the eastern Caribbean and strengthened into a hurricane Saturday night.</s>
<s docid="AP880912-0095" num="34" wdcount="25"> Maximum sustained winds were near 110 mph, with tropical-storm force winds extending up to 250 miles to the north and 100 miles to the south.</s>
<s docid="AP880912-0137" num="18" wdcount="21"> A National Weather Service report said the hurricane was moving west at 17 mph with maximum sustained winds of 115 mph.</s>
<s docid="AP880912-0137" num="22" wdcount="13"> Gilbert reached Jamaica after skirting southern Puerto Rico, Haiti and the Dominican Republic.</s>
<s docid="WSJ880912-0064" num=
